# SEBI Benchmark: Qwen 3B vs 7B (4-bit)

Tests both models on the same 50 SEBI prompts to measure:
- Prompts/sec
- Output tokens/sec
- Estimated time for 14,022 prompts
- Sample output quality

**Setup**: Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1: Check GPU
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 2: Install vLLM (pin versions that work on Colab T4)
!pip install -q vllm==0.8.5.post1 --no-deps
!pip install -q transformers==4.44.2 tokenizers>=0.19,<0.20 huggingface-hub>=0.23.2,<1.0
!pip install -q accelerate xformers ray[default]>=2.10 prometheus-fastapi-instrumentator
import vllm; print(f"vLLM {vllm.__version__}")

In [ ]:
# Cell 3: Upload sebi_colab_data.zip and extract
import zipfile, os
zip_path = "/content/sebi_colab_data.zip"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    n = len([f for f in os.listdir('/content/SEBI_Canonical/') if f.endswith('.json')])
    print(f"Extracted! {n} canonical docs found.")
else:
    print("Upload sebi_colab_data.zip first! (folder icon → Upload)")

In [ ]:
# Cell 4: Build sample prompts (loads docs, creates 50 test prompts)
import json, re, os, hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

DATA_ROOT = "/content/SEBI_Canonical"
MIN_SECTION_CHARS = 500
MAX_SECTION_CHARS = 6000
MAX_CHARS_PER_BATCH = 10000
MAX_SECTIONS_PER_BATCH = 5
SAMPLE_SIZE = 50  # prompts to benchmark

@dataclass
class Section:
    section_id: str; section_type: str; number: str; title: str; text: str
    parent_id: Optional[str] = None; parent_type: Optional[str] = None
    parent_number: Optional[str] = None; depth: int = 0

STRUCTURAL_TAGS = {"chapter","regulation","section","rule","clause","schedule","annexure","part","form","preamble","order_header","recital","direction","penalty","text"}
DEPTH_MAP = {"preamble":0,"order_header":0,"chapter":1,"part":1,"regulation":2,"section":2,"rule":2,"schedule":2,"annexure":2,"form":3,"clause":3,"recital":1,"direction":1,"penalty":1,"text":0}

def _strip_xml_tags(t): return re.sub(r'<[^>]+>', '', t).strip()
def _unescape_xml(t): return t.replace('&amp;','&').replace('&lt;','<').replace('&gt;','>').replace('&quot;','"').replace('&apos;',"'")
def _extract_number(t, st):
    m = re.match(r'^\s*(\d+[A-Z]?)\s*\.\s', t)
    if m: return m.group(1)
    m = re.match(rf'^\s*{st.capitalize()}\s+(\d+[A-Z]?)\b', t, re.IGNORECASE)
    return m.group(1) if m else ""

def parse_structured_text(st, did):
    sections = []; cc = None; idx = 0
    bm = re.search(r'<body>(.*?)</body>', st, re.DOTALL)
    body = bm.group(1) if bm else st
    tn = "|".join(STRUCTURAL_TAGS)
    op = re.compile(rf'<(?P<tag>{tn})(?P<attrs>[^>]*)>', re.IGNORECASE)
    ms = list(op.finditer(body))
    if not ms:
        t = _strip_xml_tags(body)
        if t.strip(): sections.append(Section(f"{did}/sec_0","text","","",t,depth=0))
        return sections
    for m in ms:
        tag = m.group("tag").lower(); attrs = m.group("attrs")
        nm = re.search(r'number="([^"]*)"', attrs)
        tm = re.search(r'title="([^"]*)"', attrs)
        num = nm.group(1) if nm else ""
        title = _unescape_xml(tm.group(1)) if tm else ""
        cs = m.end()
        cp = body.find(f"</{m.group('tag')}>", cs)
        ce = cp if cp != -1 else len(body)
        text = _unescape_xml(_strip_xml_tags(body[cs:ce]))
        if not text.strip(): continue
        if not num and tag in ("regulation","section","rule"): num = _extract_number(text, tag)
        if title and text.strip().startswith(title.strip()[:50]): title = ""
        sid = f"{did}/sec_{idx}"; idx += 1
        pid = pt = pn = None
        if tag in ("regulation","section","rule","clause") and cc:
            pid = cc.section_id; pt = cc.section_type; pn = cc.number
        sec = Section(sid, tag, num, title, text, pid, pt, pn, DEPTH_MAP.get(tag, 0))
        sections.append(sec)
        if tag == "chapter": cc = sec
    return sections

def split_document(doc):
    did = doc.get("document_id", "unknown")
    st = doc.get("structured_text", "")
    if not st:
        t = doc.get("text", "")
        if t.strip(): return [Section(f"{did}/sec_0","text","",doc.get("title",""),t,depth=0)]
        return []
    secs = parse_structured_text(st, did)
    f = [s for s in secs if len(s.text) >= 50]
    return f if f else secs

TASK_GROUPS = [
    ["definition", "plain_english", "summary", "applicability"],
    ["key_obligations", "compliance_checklist", "entity_understanding", "legal_citation"],
    ["scenario", "violation_detection", "cross_reference", "reasoning_chain"],
]
TASK_DESCS = {
    "definition": 'definition - Define key terms. Output: {"term","definition","citation"}',
    "plain_english": "plain_english - Explain in plain English. Output: text",
    "summary": 'summary - Four levels. Output: {"one_sentence","three_sentence","five_bullets":[],"detailed"}',
    "key_obligations": 'key_obligations - Extract obligations. Output: {"obligations":[{"entity","obligation","deadline","reporting_requirement"}]}',
    "compliance_checklist": 'compliance_checklist - Checklist. Output: {"checklist":[{"category","item","citation"}]}',
    "applicability": 'applicability - Who does this apply to? Output: {"applies_to":[],"exemptions":[],"effective_when","conditions":[],"not_applicable_when"}',
    "scenario": 'scenario - 3 scenarios (easy/medium/hard). Output: {"scenario","question","applicable_regulation","reasoning","conclusion","difficulty"}',
    "violation_detection": 'violation_detection - 3 scenarios. Output: {"violation_status","scenario","reasoning","remediation"}',
    "cross_reference": 'cross_reference - References. Output: {"cross_references":[{"referenced_document","reference_type","relationship","explanation"}]}',
    "entity_understanding": 'entity_understanding - Entity roles. Output: {"entities":[{"entity_name","entity_type","role_in_section","obligations":[]}]}',
    "legal_citation": "legal_citation - Answer with citations. Output: text",
    "reasoning_chain": 'reasoning_chain - Facts->Regs->Analysis->Conclusion. Output: {"facts","applicable_regulations":[],"analysis","conclusion","references":[]}',
}
SYS_PROMPT = """You are a SEBI regulatory expert. Generate instruction-following examples from legal document sections.
Rules: Support claims with source text. Cite regulation/section numbers. Output valid JSON only.
Tasks (Group {group_index} of {total_groups}):
{task_descriptions}"""
USR_PROMPT = """Document: {document_title} ({document_type}, {domain})

## Sections
{sections_json}

## Output Format
JSON array, each element:
{{"section_index": 0, "task_type": "...", "difficulty": "easy|medium|hard|standard", "instruction": "...", "input": "", "output": "...", "citations": []}}
Generate JSON array now."""

def render(t, v): return re.sub(r'\{\{\s*(\w+)\s*\}\}', lambda m: str(v.get(m.group(1).strip(), "")), t)
def build_sections_json(sections):
    return json.dumps([{"index": i, "section_type": s.section_type, "number": s.number, "title": s.title, "text": s.text} for i, s in enumerate(sections)], ensure_ascii=False)
def create_batches(sections):
    batches = []; cur = []; chars = 0
    for s in sections:
        t = s.text
        if len(t) > MAX_SECTION_CHARS: t = t[:MAX_SECTION_CHARS] + "\n[...]"
        if cur and (chars + len(t) > MAX_CHARS_PER_BATCH or len(cur) >= MAX_SECTIONS_PER_BATCH):
            batches.append(cur); cur = []; chars = 0
        cur.append(Section(s.section_id, s.section_type, s.number, s.title, t, s.parent_id, s.parent_type, s.parent_number, s.depth))
        chars += len(t)
    if cur: batches.append(cur)
    return batches

# Load docs and build ALL prompts
root = Path(DATA_ROOT)
docs = []
for p in sorted(root.rglob("*.json")):
    if p.name.startswith("_") or p.name == "pipeline_report.json": continue
    try:
        d = json.loads(p.read_text())
    except: continue
    if d.get("status") == "ok": docs.append(d)

all_prompts = []
for doc in docs:
    secs = [s for s in split_document(doc) if len(s.text) >= MIN_SECTION_CHARS]
    if not secs: continue
    for batch in create_batches(secs):
        sj = build_sections_json(batch)
        for gi, tg in enumerate(TASK_GROUPS):
            td = "\n".join(f"{i+1}. {TASK_DESCS.get(t, t)}" for i, t in enumerate(tg))
            v = {"document_title": doc.get("title",""), "document_type": doc.get("document_type",""), "domain": doc.get("domain",""), "group_index": str(gi+1), "total_groups": str(len(TASK_GROUPS)), "task_descriptions": td, "sections_json": sj}
            all_prompts.append({"system": render(SYS_PROMPT, v), "user": render(USR_PROMPT, v)})

print(f"Total docs: {len(docs)}")
print(f"Total prompts: {len(all_prompts)}")

# Pick SAMPLE_SIZE evenly spaced prompts for benchmarking
import random
random.seed(42)
sample_prompts = random.sample(all_prompts, min(SAMPLE_SIZE, len(all_prompts)))
print(f"Sample size: {len(sample_prompts)} prompts")

In [ ]:
# Cell 5: BENCHMARK FUNCTION — runs vLLM on sample prompts, returns stats
import time
from vllm import LLM, SamplingParams

def benchmark_model(model_id, sample_prompts, quantization=None, max_model_len=4096, gpu_mem_util=0.90):
    print(f"\n{'='*60}")
    print(f"Loading: {model_id}")
    if quantization:
        print(f"Quantization: {quantization}")
    print(f"{'='*60}")

    llm = LLM(
        model=model_id,
        quantization=quantization,
        dtype="float16",
        max_model_len=max_model_len,
        gpu_memory_utilization=gpu_mem_util,
        trust_remote_code=True,
        enforce_eager=False,
    )
    tokenizer = llm.get_tokenizer()

    # Build chat-formatted prompts
    chat_prompts = []
    for p in sample_prompts:
        msg = [{"role": "system", "content": p["system"]}, {"role": "user", "content": p["user"]}]
        chat_prompts.append(tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True))

    sampling = SamplingParams(
        temperature=0.4,
        top_p=0.9,
        max_tokens=1024,
        repetition_penalty=1.05,
    )

    print(f"Running {len(chat_prompts)} prompts...")
    t0 = time.time()
    outputs = llm.generate(chat_prompts, sampling)
    elapsed = time.time() - t0

    # Collect stats
    total_output_tokens = 0
    results = []
    for o in outputs:
        out_text = o.outputs[0].text
        tok_count = len(o.outputs[0].token_ids)
        total_output_tokens += tok_count
        results.append(out_text)

    prompts_per_sec = len(chat_prompts) / elapsed
    tokens_per_sec = total_output_tokens / elapsed
    est_total_time = len(all_prompts) / prompts_per_sec / 60  # minutes for 14k

    print(f"\n{'='*60}")
    print(f"BENCHMARK RESULTS: {model_id}")
    print(f"{'='*60}")
    print(f"  Prompts:          {len(chat_prompts)}")
    print(f"  Time:              {elapsed:.1f}s ({elapsed/60:.1f}m)")
    print(f"  Prompts/sec:       {prompts_per_sec:.2f}")
    print(f"  Output tokens/sec: {tokens_per_sec:.0f}")
    print(f"  Avg tokens/prompt: {total_output_tokens // len(chat_prompts)}")
    print(f"  Est. time for {len(all_prompts)} prompts: {est_total_time:.0f} min ({est_total_time/60:.1f}h)")
    print(f"{'='*60}")

    # Show 2 sample outputs
    print(f"\n--- SAMPLE OUTPUT 1 ---")
    print(results[0][:600])
    print(f"\n--- SAMPLE OUTPUT 2 ---")
    print(results[1][:600])

    # Free VRAM
    del llm
    import gc, torch
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "model": model_id,
        "prompts": len(chat_prompts),
        "time_sec": elapsed,
        "prompts_per_sec": prompts_per_sec,
        "tokens_per_sec": tokens_per_sec,
        "est_total_min": est_total_time,
        "outputs": results,
    }

In [ ]:
# Cell 6: BENCHMARK 3B (float16) — ~5 min
results_3b = benchmark_model(
    model_id="Qwen/Qwen2.5-3B-Instruct",
    sample_prompts=sample_prompts,
    quantization=None,
    max_model_len=4096,
    gpu_mem_util=0.90,
)

In [ ]:
# Cell 7: BENCHMARK 7B (AWQ 4-bit) — ~7 min
results_7b = benchmark_model(
    model_id="Qwen/Qwen2.5-7B-Instruct-AWQ",
    sample_prompts=sample_prompts,
    quantization="awq",
    max_model_len=4096,
    gpu_mem_util=0.90,
)

In [ ]:
# Cell 8: COMPARISON TABLE
print(f"\n{'='*70}")
print(f"COMPARISON: 3B vs 7B on {len(sample_prompts)} SEBI prompts")
print(f"{'='*70}")
print(f"{'Metric':<25} {'3B (float16)':>20} {'7B (AWQ 4-bit)':>20}")
print(f"{'-'*65}")
print(f"{'Time':.<25} {results_3b['time_sec']:>18.1f}s {results_7b['time_sec']:>18.1f}s")
print(f"{'Prompts/sec':.<25} {results_3b['prompts_per_sec']:>18.2f} {results_7b['prompts_per_sec']:>18.2f}")
print(f"{'Tokens/sec':.<25} {results_3b['tokens_per_sec']:>18.0f} {results_7b['tokens_per_sec']:>18.0f}")
print(f"{'Est. time (14k prompts)':.<25} {results_3b['est_total_min']:>16.0f} min {results_7b['est_total_min']:>16.0f} min")
print(f"{'Est. time (14k prompts)':.<25} {results_3b['est_total_min']/60:>16.1f} h  {results_7b['est_total_min']/60:>16.1f} h")
print(f"{'-'*65}")

# Quality check: count valid JSON outputs
def count_valid_json(outputs):
    valid = 0
    for o in outputs:
        try:
            p = json.loads(o)
            valid += 1
        except:
            m = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', o, re.DOTALL)
            if m:
                try: json.loads(m.group(1)); valid += 1
                except: pass
            else:
                s = o.find('['); e = o.rfind(']')
                if s != -1 and e > s:
                    try: json.loads(o[s:e+1]); valid += 1
                    except: pass
    return valid

v3 = count_valid_json(results_3b['outputs'])
v7 = count_valid_json(results_7b['outputs'])
print(f"{'Valid JSON outputs':.<25} {v3:>13}/{len(sample_prompts):<4} {v7:>13}/{len(sample_prompts):<4}")
print(f"{'JSON success rate':.<25} {v3/len(sample_prompts)*100:>17.1f}% {v7/len(sample_prompts)*100:>17.1f}%")
print(f"{'='*70}")

In [ ]:
# Cell 9: Side-by-side output comparison (same prompt)
idx = 0  # change to see different prompts

print(f"{'='*70}")
print(f"PROMPT {idx} — System prompt (first 200 chars):")
print(f"{sample_prompts[idx]['system'][:200]}...")
print(f"{'='*70}")

print(f"\n{'='*70}")
print(f"3B OUTPUT:")
print(f"{'='*70}")
print(results_3b['outputs'][idx][:800])

print(f"\n{'='*70}")
print(f"7B OUTPUT:")
print(f"{'='*70}")
print(results_7b['outputs'][idx][:800])